In [21]:
# ==========================================
# 1. IMPORTAÇÕES E CONFIGURAÇÃO DE RENDER
# ==========================================
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Configura o renderizador para 'colab' / 'notebook_connected'
pio.renderers.default = "colab"

print("⚡ Carregando base de dados otimizada...")

# ==========================================
# 2. CARREGAMENTO RÁPIDO DOS DADOS
# ==========================================
url = "https://raw.githubusercontent.com/wcota/covid19br/master/cases-brazil-states.csv"
df = pd.read_csv(url, usecols=['date', 'state', 'newCases', 'newDeaths'])
df = df[df['state'] != 'TOTAL'].copy()

# Mapeamentos Geográficos
coords = {
    'AC': (-9.02, -70.81), 'AL': (-9.57, -36.78), 'AM': (-3.41, -65.85), 'AP': (1.41, -51.77),
    'BA': (-12.96, -38.51), 'CE': (-5.20, -39.53), 'DF': (-15.78, -47.92), 'ES': (-19.18, -40.30),
    'GO': (-15.82, -49.83), 'MA': (-5.42, -45.44), 'MG': (-18.51, -44.55), 'MS': (-20.77, -54.78),
    'MT': (-12.68, -56.92), 'PA': (-1.99, -54.93), 'PB': (-7.23, -36.78), 'PE': (-8.28, -35.03),
    'PI': (-7.71, -42.72), 'PR': (-25.25, -52.02), 'RJ': (-22.84, -43.15), 'RN': (-5.81, -36.59),
    'RO': (-11.50, -63.58), 'RR': (2.73, -61.32), 'RS': (-30.03, -51.23), 'SC': (-27.24, -50.21),
    'SE': (-10.57, -37.38), 'SP': (-23.55, -46.63), 'TO': (-10.18, -48.33)
}

regioes = {
    'SP': 'Sudeste', 'RJ': 'Sudeste', 'MG': 'Sudeste', 'ES': 'Sudeste',
    'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
    'BA': 'Nordeste', 'PE': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste',
    'PB': 'Nordeste', 'RN': 'Nordeste', 'AL': 'Nordeste', 'SE': 'Nordeste', 'PI': 'Nordeste',
    'AM': 'Norte', 'PA': 'Norte', 'AP': 'Norte', 'RR': 'Norte', 'RO': 'Norte', 'AC': 'Norte', 'TO': 'Norte',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste'
}

df['date'] = pd.to_datetime(df['date'])
df['ano'] = df['date'].dt.year
df['mes_ano'] = df['date'].dt.to_period('M').astype(str)
df['regiao'] = df['state'].map(regioes)
df['lat'] = df['state'].apply(lambda x: coords.get(x, (0,0))[0])
df['lon'] = df['state'].apply(lambda x: coords.get(x, (0,0))[1])

# Agrupamento de Dados
df_monthly = df.groupby(['ano', 'mes_ano', 'regiao', 'state', 'lat', 'lon'], as_index=False).agg({
    'newCases': 'sum',
    'newDeaths': 'sum'
})

print("✅ Dados prontos!")

# ==========================================
# 3. EXIBIÇÃO DIRETA DOS COMPONENTES
# ==========================================

# --- REQUISITO 1: TABELA DE DADOS ---
print("\n--- 📋 1. TABELA DE DADOS RESUMIDA ---")
df_tabela = df_monthly.groupby(['regiao', 'state'], as_index=False).agg({
    'newCases': 'sum', 'newDeaths': 'sum'
}).rename(columns={'regiao': 'Região', 'state': 'UF', 'newCases': 'Total Casos', 'newDeaths': 'Total Óbitos'})

display(df_tabela.head(10))

# --- REQUISITO 2: SÉRIE TEMPORAL (GRÁFICO DE LINHA) ---
print("\n--- 📈 2. SÉRIE TEMPORAL: EVOLUÇÃO MENSAL ---")
df_time = df_monthly.groupby('mes_ano', as_index=False)['newCases'].sum()

fig_line = px.line(
    df_time, x='mes_ano', y='newCases',
    title='Evolução Mensal de Novos Casos no Brasil',
    labels={'mes_ano': 'Mês/Ano', 'newCases': 'Novos Casos'},
    markers=True, template='plotly_white'
)
fig_line.update_traces(line_color='#1f77b4', line_width=3)
fig_line.update_layout(xaxis_tickangle=-45, height=400)
fig_line.show(renderer="colab")

# --- REQUISITO 3: VISUALIZAÇÃO GEOGRÁFICA (MAPA) ---
print("\n--- 🗺️ 3. VISUALIZAÇÃO GEOGRÁFICA (MAPA DE CONCENTRAÇÃO) ---")
df_mapa = df_monthly.groupby(['state', 'lat', 'lon', 'regiao'], as_index=False)['newCases'].sum()

fig_map = px.scatter_geo(
    df_mapa, lat='lat', lon='lon', size='newCases', color='newCases',
    hover_name='state', hover_data={'regiao': True, 'newCases': ':,', 'lat': False, 'lon': False},
    size_max=35, projection="mercator", color_continuous_scale="Reds",
    title="Distribuição e Concentração Geográfica dos Casos por Estado"
)
fig_map.update_geos(
    scope='south america', center=dict(lat=-14.235, lon=-51.925),
    projection_scale=3.8, showcountries=True, showland=True, landcolor="#f8f9fa"
)
fig_map.update_layout(height=450, margin=dict(l=0, r=0, t=40, b=0))
fig_map.show(renderer="colab")

⚡ Carregando base de dados otimizada...
✅ Dados prontos!

--- 📋 1. TABELA DE DADOS RESUMIDA ---


,Região,UF,Total Casos,Total Óbitos
0,Centro-Oeste,DF,898072,11845
1,Centro-Oeste,GO,1900459,28041
2,Centro-Oeste,MS,607776,11016
3,Centro-Oeste,MT,879415,15090
4,Nordeste,AL,337568,7242
5,Nordeste,BA,1792572,31459
6,Nordeste,CE,1453417,28157
7,Nordeste,MA,493418,11053
8,Nordeste,PB,707854,10542
9,Nordeste,PE,1157160,22687



--- 📈 2. SÉRIE TEMPORAL: EVOLUÇÃO MENSAL ---



--- 🗺️ 3. VISUALIZAÇÃO GEOGRÁFICA (MAPA DE CONCENTRAÇÃO) ---
